In [1]:
from langfuse import get_client
import dotenv 
dotenv.load_dotenv()  # Load environment variables from .env file

langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

from pydantic_ai.agent import Agent

# Initialize Pydantic AI instrumentation
Agent.instrument_all()

Langfuse client is authenticated and ready!


In [2]:
from pydantic_ai import Agent, RunContext

roulette_agent = Agent(
    'openrouter:anthropic/claude-sonnet-4-6',
    deps_type=int,
    system_prompt=(
        'Use the `roulette_wheel` function to see if the '
        'customer has won based on the number they provide.'
    ),
    instrument=True
)

@roulette_agent.tool
async def roulette_wheel(ctx: RunContext[int], square: int) -> str:
    """check if the square is a winner"""
    return 'winner' if square == ctx.deps else 'loser'

Demo of observe in langfuse.

In [3]:
from langfuse import observe

@observe(name="roulette_round")
async def play_round(user_guess: int, winning_number: int) -> str:
    result = await roulette_agent.run(
        f"My number is {user_guess}. Did I win?",
        deps=winning_number,
    )
    return result.output

@observe(name="roulette_session")
async def play_session(rounds: list[tuple[int, int]]) -> list[str]:
    outputs: list[str] = []
    for guess, winner in rounds:
        outcome = await play_round(user_guess=guess, winning_number=winner)
        line = f"guess={guess}, winner={winner} -> {outcome}"
        print(line)
        outputs.append(line)
    return outputs

# A single observed call creates one trace with nested operations.
rounds = [
    (7, 7),   # winner
    (13, 7),  # loser
    (22, 22), # winner
]

_ = await play_session(rounds)
print("Open Langfuse and filter by trace/operation name: roulette_session")

guess=7, winner=7 -> 🎉 Congratulations, you're a **winner**! 🎉 Your number, **7**, is the lucky number on the roulette wheel! Well done! 🍀
guess=13, winner=7 -> Unfortunately, it looks like **13** is not a winning number this time. Better luck next time! 🍀 Would you like to try another number?
guess=22, winner=22 -> 🎉 Congratulations, you're a **winner**! 🎉 The roulette wheel landed on **22** and that's a winning square! Well done! 🥳
Open Langfuse and filter by trace/operation name: roulette_session
